[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_08/notebook_8_4_llm_prompting.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# LLM Prompting Strategies for Clinical Tasks

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Design effective prompts** for zero-shot and few-shot learning
2. **Implement chain-of-thought reasoning** for complex clinical tasks
3. **Use LLM APIs** (OpenRouter, HuggingFace, Google Vertex AI) for clinical NLP
4. **Parse and validate LLM outputs** for structured data extraction
5. **Compare LLM prompting vs. fine-tuned models** on clinical tasks

## Clinical Context

**Clinical Scenario**: Automated differential diagnosis generation from patient presentations:

**Input**: "65-year-old male with chest pain, dyspnea, elevated troponin"

**Desired Output**:
1. Acute myocardial infarction (high probability)
2. Pulmonary embolism (moderate probability)
3. Aortic dissection (low probability)

**Why LLMs?**
- **No training data required**: Zero-shot learning on new tasks
- **Rapid prototyping**: Test ideas in minutes vs. weeks
- **Reasoning capabilities**: Chain-of-thought for complex clinical logic
- **Flexibility**: Single model for multiple tasks (classification, generation, extraction)

**Challenges**:
- **Hallucinations**: May generate plausible but incorrect information
- **Inconsistency**: Same prompt may yield different outputs
- **Cost**: $0.15-0.60 per 1K requests vs. $0 for local models
- **Latency**: 1-3 seconds vs. <100ms for BERT

---

## Getting Started: API Access

### Option 1: OpenRouter (Recommended for Students)

**Why OpenRouter?**
- Access to multiple LLM providers (OpenAI, Anthropic, Google, Meta) through single API
- Pay-as-you-go pricing (no subscription required)
- **Free credits**: $1-5 for new accounts

**Registration Steps**:
1. Visit: https://openrouter.ai/
2. Click "Sign Up" (top right)
3. Sign in with Google, GitHub, or email
4. Go to "Keys" tab → "Create Key"
5. Copy your API key (starts with `sk-or-v1-...`)
6. **Free credits**: New accounts receive $1-5 in credits (~200-1000 requests)

**Cost Example**:
- Claude 3.5 Haiku: ~$0.25 per 1M input tokens (~$0.00025 per request)
- GPT-4o Mini: ~$0.15 per 1M input tokens
- Your $1 credit ≈ 4,000 requests with Haiku

### Option 2: HuggingFace Inference API

**Registration Steps**:
1. Visit: https://huggingface.co/
2. Click "Sign Up" (top right)
3. Verify email
4. Go to Settings → Access Tokens
5. Create "Read" token
6. **Free tier**: Rate-limited but sufficient for learning

**Available Models**:
- Mistral-7B, Llama-3-8B (smaller, faster, free)
- BioGPT, ClinicalBERT (specialized medical models)

### Option 3: Google Vertex AI

**Registration Steps**:
1. Visit: https://cloud.google.com/vertex-ai
2. Click "Try it free" → Sign in with Google account
3. **$300 free credits** for 90 days (new users)
4. Enable Vertex AI API
5. Create service account → Download JSON key

**Models**:
- Gemini Pro: General purpose
- Gemini Pro Vision: Multimodal (text + images)
- PaLM 2: Text generation

---

### Installation

```bash
# For OpenRouter (uses OpenAI-compatible API)
pip install openai

# For HuggingFace
pip install huggingface_hub

# For Google Vertex AI
pip install google-cloud-aiplatform
```

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
from typing import List, Dict, Optional
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported successfully!")
print("\n⚠️  Note: To run LLM examples, you'll need API keys.")
print("   See 'Getting Started: API Access' section above for registration instructions.")

## 1. Simulated LLM Interface

For this educational notebook, we'll create a **simulated LLM** that demonstrates prompting concepts without requiring API keys.

**Real LLM implementation** is shown in subsequent sections.

In [ ]:
class SimulatedLLM:
    """
    Simulated LLM for educational purposes.

    Returns pre-defined responses based on keywords in prompts.
    Real LLMs (GPT-4, Claude, Gemini) use transformer models.
    """

    def __init__(self, model_name="simulated-medical-llm"):
        self.model_name = model_name
        self.response_count = 0

        # Pre-defined clinical knowledge
        self.clinical_patterns = {
            'chest pain + troponin': 'Acute myocardial infarction',
            'dyspnea + wheezing': 'Asthma exacerbation',
            'fever + cough + infiltrate': 'Pneumonia',
            'hyperglycemia + ketones': 'Diabetic ketoacidosis',
            'weakness + aphasia': 'Acute ischemic stroke',
            'tremor + rigidity': "Parkinson's disease",
        }

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 500) -> str:
        """
        Simulate LLM text generation.

        Args:
            prompt: Input prompt
            temperature: Controls randomness (0=deterministic, 1=creative)
            max_tokens: Maximum response length

        Returns:
            Generated text response
        """
        self.response_count += 1
        prompt_lower = prompt.lower()

        # Detect task type from prompt
        if 'classify' in prompt_lower or 'specialty' in prompt_lower:
            return self._classify_specialty(prompt)
        elif 'differential' in prompt_lower or 'diagnos' in prompt_lower:
            return self._generate_differential(prompt)
        elif 'extract' in prompt_lower:
            return self._extract_entities(prompt)
        elif 'summarize' in prompt_lower:
            return self._summarize_note(prompt)
        else:
            return "I understand you're asking about a clinical scenario. Could you please specify the task (classify, diagnose, extract, or summarize)?"

    def _classify_specialty(self, prompt: str) -> str:
        """Simulate specialty classification."""
        prompt_lower = prompt.lower()

        if 'chest pain' in prompt_lower or 'troponin' in prompt_lower or 'heart' in prompt_lower:
            return "cardiology"
        elif 'dyspnea' in prompt_lower or 'wheezing' in prompt_lower or 'spirometry' in prompt_lower:
            return "pulmonology"
        elif 'glucose' in prompt_lower or 'diabetes' in prompt_lower or 'a1c' in prompt_lower:
            return "endocrinology"
        elif 'weakness' in prompt_lower or 'seizure' in prompt_lower or 'stroke' in prompt_lower:
            return "neurology"
        else:
            return "general_medicine"

    def _generate_differential(self, prompt: str) -> str:
        """Simulate differential diagnosis generation."""
        prompt_lower = prompt.lower()

        # Pattern matching for common presentations
        for pattern, diagnosis in self.clinical_patterns.items():
            keywords = pattern.split(' + ')
            if all(kw in prompt_lower for kw in keywords):
                return f"""Based on the clinical presentation, the differential diagnosis includes:

1. **{diagnosis}** (High probability)
   - Primary diagnosis given the constellation of symptoms
   - Requires urgent evaluation and management

2. **Alternative diagnosis** (Moderate probability)
   - Consider if initial diagnosis is ruled out
   - Additional testing may be needed

3. **Less likely diagnosis** (Low probability)
   - Keep on differential but less likely given presentation
   - Consider if patient not responding to initial treatment
"""

        return """Based on the clinical presentation, consider:
1. Most likely diagnosis (given key symptoms)
2. Alternative diagnosis (if tests rule out #1)
3. Less likely but important not to miss
"""

    def _extract_entities(self, prompt: str) -> str:
        """Simulate entity extraction."""
        # Simple keyword extraction
        entities = {
            'symptoms': [],
            'tests': [],
            'medications': []
        }

        symptoms = ['chest pain', 'dyspnea', 'fever', 'cough', 'weakness']
        tests = ['troponin', 'ECG', 'chest x-ray', 'CT', 'MRI']
        meds = ['aspirin', 'metformin', 'lisinopril', 'furosemide']

        prompt_lower = prompt.lower()
        for symptom in symptoms:
            if symptom in prompt_lower:
                entities['symptoms'].append(symptom)
        for test in tests:
            if test in prompt_lower:
                entities['tests'].append(test)
        for med in meds:
            if med in prompt_lower:
                entities['medications'].append(med)

        return json.dumps(entities, indent=2)

    def _summarize_note(self, prompt: str) -> str:
        """Simulate clinical note summarization."""
        return """**Summary:**
Patient presentation: [Key symptoms and findings]
Assessment: [Primary diagnosis]
Plan: [Management approach]
"""

# Initialize simulated LLM
sim_llm = SimulatedLLM()

print("✓ Simulated LLM initialized")
print(f"  Model: {sim_llm.model_name}")
print("\n💡 This simulated LLM demonstrates prompting concepts.")
print("   For real applications, use actual LLM APIs (see production examples below).")

## 2. Zero-Shot Learning

**Zero-shot**: LLM performs task with NO examples, only task instructions.

**Advantages**:
- No training data required
- Fastest to implement
- Works for novel tasks

**Disadvantages**:
- Lower accuracy than few-shot or fine-tuned
- May not follow specific output format
- Sensitive to prompt wording

In [ ]:
# Zero-shot classification example
zero_shot_prompt = """Classify the following clinical case into one specialty: cardiology, pulmonology, endocrinology, or neurology.

Clinical case: Patient with chest pain radiating to left arm. ECG shows ST elevation. Troponin elevated at 0.8 ng/mL.

Specialty:"""

response = sim_llm.generate(zero_shot_prompt)

print("Zero-Shot Classification Example:")
print("="*70)
print("\nPrompt:")
print(zero_shot_prompt)
print("\nResponse:")
print(response)

# Test multiple cases
test_cases = [
    "Patient with dyspnea and wheezing. Spirometry shows FEV1/FVC <0.70.",
    "Fasting glucose 180 mg/dL, HbA1c 8.5%, polyuria present.",
    "Sudden onset right-sided weakness and aphasia. CT shows hypodensity.",
]

print("\n" + "="*70)
print("Zero-Shot Performance on Multiple Cases:")
print("="*70)

results = []
for i, case in enumerate(test_cases, 1):
    prompt = f"""Classify into one specialty: cardiology, pulmonology, endocrinology, or neurology.

Clinical case: {case}

Specialty:"""

    response = sim_llm.generate(prompt)
    results.append({'case': case[:60] + '...', 'specialty': response})
    print(f"\nCase {i}: {case[:60]}...")
    print(f"Predicted: {response}")

# Create results DataFrame
df_results = pd.DataFrame(results)
print("\n" + "="*70)
print(df_results.to_string(index=False))

## 3. Few-Shot Learning

**Few-shot**: LLM learns from 2-10 examples provided in the prompt.

**Advantages**:
- Better accuracy than zero-shot (+5-15%)
- Teaches specific output format
- Shows edge cases and reasoning

**Disadvantages**:
- Longer prompts (higher cost)
- Need to carefully select representative examples
- Still less accurate than fine-tuning on large datasets

In [ ]:
# Few-shot classification with examples
few_shot_prompt = """Classify clinical cases into specialties: cardiology, pulmonology, endocrinology, or neurology.

Examples:

Case: Patient with palpitations. EKG shows irregular rhythm with atrial fibrillation.
Specialty: cardiology

Case: Fever and cough. Chest X-ray shows right lower lobe infiltrate.
Specialty: pulmonology

Case: Weight gain, fatigue, and cold intolerance. TSH elevated at 12 mIU/L.
Specialty: endocrinology

Case: Progressive memory loss. MRI shows cortical atrophy. MMSE score 18/30.
Specialty: neurology

Now classify this case:

Case: Patient with chest pain radiating to left arm. ECG shows ST elevation. Troponin elevated.
Specialty:"""

response = sim_llm.generate(few_shot_prompt)

print("Few-Shot Classification Example:")
print("="*70)
print("\nPrompt structure:")
print("  1. Task description")
print("  2. 4 labeled examples (one per specialty)")
print("  3. New case to classify")
print("\nResponse:")
print(response)

# Compare zero-shot vs few-shot
print("\n" + "="*70)
print("Zero-Shot vs. Few-Shot Comparison:")
print("="*70)
print("\n**Zero-shot**: Task description only")
print("  - Faster (shorter prompt)")
print("  - Less accurate")
print("  - May not follow exact format")
print("\n**Few-shot**: Task description + examples")
print("  - Slower (longer prompt = higher cost)")
print("  - More accurate (+5-15%)")
print("  - Learns output format from examples")
print("\n**Typical accuracy on specialty classification**:")
print("  - Zero-shot: 75-82%")
print("  - Few-shot (4 examples): 82-88%")
print("  - Fine-tuned BERT: 88-92%")

## 4. Chain-of-Thought (CoT) Reasoning

**Chain-of-Thought**: Prompt LLM to show its reasoning steps before final answer.

**Key phrase**: "Let's think step by step"

**Benefits**:
- Improves accuracy on complex reasoning tasks (+10-20%)
- Makes reasoning transparent (audit trail)
- Helps catch logical errors
- Better for multi-step clinical decision making

**When to use**:
- Differential diagnosis (requires weighing evidence)
- Drug interaction checking (multi-step logic)
- Clinical guideline application (conditional logic)

In [ ]:
# Chain-of-thought for differential diagnosis
cot_prompt = """Generate a differential diagnosis for the following case. Let's think step by step.

Patient: 65-year-old male
Chief complaint: Acute chest pain
History:
  - Pain started 2 hours ago
  - Radiates to left arm
  - Associated with dyspnea and diaphoresis
Exam:
  - BP 160/95, HR 105, RR 22
  - Diaphoretic, anxious
Labs:
  - Troponin I: 0.8 ng/mL (elevated)
  - D-dimer: normal
ECG: ST elevation in leads II, III, aVF

Let's analyze this step by step to generate a differential diagnosis:"""

response = sim_llm.generate(cot_prompt)

print("Chain-of-Thought Reasoning Example:")
print("="*70)
print("\nPrompt:")
print(cot_prompt)
print("\nResponse (with reasoning):")
print(response)

# Standard vs. CoT comparison
print("\n" + "="*70)
print("Standard Prompt vs. Chain-of-Thought:")
print("="*70)

standard_prompt = """Generate a differential diagnosis for:
65-year-old male with chest pain, elevated troponin, ST elevation on ECG.

Differential diagnosis:"""

cot_prompt_short = """Generate a differential diagnosis for:
65-year-old male with chest pain, elevated troponin, ST elevation on ECG.

Let's think through this step by step:"""

standard_response = sim_llm.generate(standard_prompt)
cot_response = sim_llm.generate(cot_prompt_short)

print("\n**Without CoT**:")
print(standard_response[:150] + "...")
print("\n**With CoT** ('Let's think step by step'):")
print(cot_response[:150] + "...")

print("\n💡 Key insight: CoT prompting encourages explicit reasoning,")
print("   making the model's logic transparent and auditable.")

## 5. Structured Output Parsing

**Challenge**: LLMs generate free text, but we often need structured data.

**Solution**:
1. Request specific format (JSON, bullet points, etc.)
2. Parse and validate output
3. Retry on failures

In [ ]:
class StructuredOutputParser:
    """
    Parse and validate LLM outputs into structured format.
    """

    @staticmethod
    def parse_json(response: str) -> Optional[Dict]:
        """
        Extract JSON from LLM response.

        Handles:
        - JSON in markdown code blocks
        - JSON with surrounding text
        - Malformed JSON
        """
        try:
            # Try direct JSON parsing
            return json.loads(response)
        except json.JSONDecodeError:
            # Try extracting from markdown code block
            json_match = re.search(r'```(?:json)?\s*({[^`]+})\s*```', response, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group(1))
                except json.JSONDecodeError:
                    pass

            # Try finding JSON object anywhere in text
            json_match = re.search(r'{[^{}]*(?:{[^{}]*}[^{}]*)*}', response, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group(0))
                except json.JSONDecodeError:
                    pass

        return None

    @staticmethod
    def validate_differential(data: Dict) -> bool:
        """
        Validate differential diagnosis structure.

        Expected format:
        {
            "diagnoses": [
                {"name": "...", "probability": "high|moderate|low", "reasoning": "..."}
            ]
        }
        """
        if not isinstance(data, dict):
            return False

        if 'diagnoses' not in data:
            return False

        diagnoses = data['diagnoses']
        if not isinstance(diagnoses, list) or len(diagnoses) == 0:
            return False

        for dx in diagnoses:
            if not all(key in dx for key in ['name', 'probability']):
                return False
            if dx['probability'] not in ['high', 'moderate', 'low']:
                return False

        return True

# Example: Request structured JSON output
structured_prompt = """Generate a differential diagnosis in JSON format:

Case: 65-year-old with chest pain, elevated troponin, ST elevation on ECG.

Return JSON with this structure:
{
  "diagnoses": [
    {
      "name": "diagnosis name",
      "probability": "high|moderate|low",
      "reasoning": "brief explanation"
    }
  ]
}

JSON:"""

# Simulate structured response
simulated_json_response = """{
  "diagnoses": [
    {
      "name": "Acute myocardial infarction (inferior wall)",
      "probability": "high",
      "reasoning": "ST elevation in inferior leads + elevated troponin strongly suggests inferior MI"
    },
    {
      "name": "Pulmonary embolism",
      "probability": "low",
      "reasoning": "Chest pain and dyspnea present but normal D-dimer makes PE less likely"
    }
  ]
}"""

print("Structured Output Example:")
print("="*70)
print("\nPrompt requests JSON format with specific structure.")
print("\nSimulated LLM Response:")
print(simulated_json_response)

# Parse and validate
parser = StructuredOutputParser()
parsed_data = parser.parse_json(simulated_json_response)

print("\n" + "="*70)
print("Parsing and Validation:")
print("="*70)

if parsed_data:
    print("\n✓ Successfully parsed JSON")
    print(f"  Number of diagnoses: {len(parsed_data['diagnoses'])}")

    is_valid = parser.validate_differential(parsed_data)
    if is_valid:
        print("✓ Structure validated")
        print("\nParsed diagnoses:")
        for i, dx in enumerate(parsed_data['diagnoses'], 1):
            print(f"\n{i}. {dx['name']} ({dx['probability']} probability)")
            print(f"   Reasoning: {dx['reasoning']}")
    else:
        print("✗ Invalid structure")
else:
    print("✗ Failed to parse JSON")

print("\n" + "="*70)
print("Best Practices for Structured Output:")
print("="*70)
print("1. Explicitly request format (JSON, YAML, etc.)")
print("2. Provide example output in prompt")
print("3. Parse with error handling")
print("4. Validate structure")
print("5. Retry on failures (up to 3 times)")
print("6. Use temperature=0 for consistent formatting")

## 6. Production Implementation Examples

### Option 1: OpenRouter API

In [ ]:
# OpenRouter Implementation
# pip install openai

"""
import openai
import os

# Configure OpenRouter
openai.api_base = "https://openrouter.ai/api/v1"
openai.api_key = os.getenv("OPENROUTER_API_KEY")  # Set in environment

def classify_specialty_openrouter(case_text: str) -> str:
    """
    Classify clinical case using OpenRouter.

    Cost: ~$0.00025 per request with Claude Haiku
    """
    prompt = f"""Classify into one specialty: cardiology, pulmonology, endocrinology, or neurology.

Clinical case: {case_text}

Specialty:"""

    response = openai.ChatCompletion.create(
        model="anthropic/claude-3-5-haiku",  # Fast + cheap
        # Alternative models:
        # "openai/gpt-4o-mini" - Good balance
        # "google/gemini-pro" - Google's model
        # "meta-llama/llama-3-8b" - Open source
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,  # Deterministic
        max_tokens=50
    )

    specialty = response.choices[0].message.content.strip().lower()
    return specialty

# Example usage
case = "Patient with chest pain and elevated troponin. ECG shows ST elevation."
specialty = classify_specialty_openrouter(case)
print(f"Classified as: {specialty}")

# Cost tracking
print(f"Estimated cost: ${response.usage.total_tokens * 0.25 / 1e6:.6f}")
"""

print("OpenRouter API Example (code above):")
print("="*70)
print("\n📋 Setup Instructions:")
print("1. Register at https://openrouter.ai/")
print("2. Get API key from Keys tab")
print("3. Set environment variable:")
print("   export OPENROUTER_API_KEY='sk-or-v1-...'")
print("4. Run code above")
print("\n💰 Cost Example (Claude Haiku):")
print("  - Input: $0.25 per 1M tokens")
print("  - Output: $1.25 per 1M tokens")
print("  - Typical request: ~200 tokens = $0.00025")
print("  - Your $1 credit ≈ 4,000 requests")

### Option 2: HuggingFace Inference API

In [ ]:
# HuggingFace Inference API
# pip install huggingface_hub

"""
from huggingface_hub import InferenceClient
import os

# Initialize client
client = InferenceClient(token=os.getenv("HF_TOKEN"))

def classify_specialty_hf(case_text: str) -> str:
    """
    Classify using HuggingFace Inference API.

    Free tier: Rate-limited but sufficient for learning
    """
    prompt = f"""Classify into: cardiology, pulmonology, endocrinology, or neurology.

Case: {case_text}

Specialty:"""

    # Using Mistral-7B (free, fast)
    response = client.text_generation(
        prompt,
        model="mistralai/Mistral-7B-Instruct-v0.2",
        max_new_tokens=50,
        temperature=0.1
    )

    specialty = response.strip().lower()
    # Parse first word (model may generate explanation)
    specialty = specialty.split()[0].strip('.,;:')

    return specialty

# Alternative: Use specialized medical models
def generate_differential_hf(case_text: str) -> str:
    """
    Generate differential using BioGPT (medical-specific model).
    """
    prompt = f"Generate differential diagnosis for: {case_text}"

    response = client.text_generation(
        prompt,
        model="microsoft/BioGPT",  # Trained on PubMed
        max_new_tokens=200,
        temperature=0.7
    )

    return response

# Example
case = "65-year-old with chest pain, elevated troponin, ST elevation."
specialty = classify_specialty_hf(case)
print(f"Classified as: {specialty}")

differential = generate_differential_hf(case)
print(f"\nDifferential:\n{differential}")
"""

print("HuggingFace API Example (code above):")
print("="*70)
print("\n📋 Setup Instructions:")
print("1. Register at https://huggingface.co/")
print("2. Go to Settings → Access Tokens")
print("3. Create 'Read' token")
print("4. Set environment variable:")
print("   export HF_TOKEN='hf_...'")
print("5. Run code above")
print("\n🎓 Free Tier:")
print("  - Rate limited (1 request/second)")
print("  - Sufficient for learning and prototyping")
print("  - Upgrade for production use")
print("\n🏥 Medical Models Available:")
print("  - BioGPT (Microsoft, PubMed-trained)")
print("  - BioBERT (NER and classification)")
print("  - Clinical-Longformer (long documents)")

### Option 3: Google Vertex AI

In [ ]:
# Google Vertex AI
# pip install google-cloud-aiplatform

"""
import vertexai
from vertexai.preview.generative_models import GenerativeModel

# Initialize Vertex AI
vertexai.init(project="YOUR_PROJECT_ID", location="us-central1")

def classify_specialty_vertex(case_text: str) -> str:
    """
    Classify using Google Gemini Pro.

    New users: $300 free credits for 90 days
    """
    model = GenerativeModel("gemini-1.5-pro")

    prompt = f"""Classify this clinical case into one specialty: cardiology, pulmonology, endocrinology, or neurology.

Case: {case_text}

Respond with only the specialty name (lowercase).

Specialty:"""

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0,
            "max_output_tokens": 50,
        }
    )

    specialty = response.text.strip().lower()
    return specialty

# Multimodal example (text + image)
from vertexai.preview.generative_models import Part

def analyze_chest_xray(image_path: str, clinical_context: str) -> str:
    """
    Analyze chest X-ray with clinical context using Gemini Pro Vision.
    """
    model = GenerativeModel("gemini-1.5-pro-vision")

    # Load image
    image = Part.from_uri(image_path, mime_type="image/jpeg")

    prompt = f"""Analyze this chest X-ray given the following clinical context:

{clinical_context}

Describe key findings and suggest differential diagnoses."""

    response = model.generate_content([prompt, image])
    return response.text

# Example
case = "Patient with chest pain and elevated troponin."
specialty = classify_specialty_vertex(case)
print(f"Classified as: {specialty}")
"""

print("Google Vertex AI Example (code above):")
print("="*70)
print("\n📋 Setup Instructions:")
print("1. Visit https://cloud.google.com/vertex-ai")
print("2. Click 'Try it free' ($300 credits for 90 days)")
print("3. Create project")
print("4. Enable Vertex AI API")
print("5. Install: gcloud auth application-default login")
print("6. Run code above")
print("\n💰 Pricing (after free credits):")
print("  - Gemini Pro: $0.25 per 1M input chars")
print("  - Gemini Pro Vision: $0.25 per 1M chars + $2.50/image")
print("\n🎯 Unique Features:")
print("  - Multimodal (text + images)")
print("  - Long context window (32K tokens)")
print("  - Google search integration")

## 7. Prompt Engineering Best Practices

### General Principles

1. **Be Specific**: Clear task definition beats vague instructions
   - ❌ "Analyze this case"
   - ✅ "Classify into cardiology, pulmonology, endocrinology, or neurology"

2. **Provide Context**: Background improves accuracy
   - ✅ "You are an experienced clinician. Review this case..."

3. **Request Format**: Specify exact output structure
   - ✅ "Respond with JSON: {\"specialty\": \"...\", \"confidence\": ...}"

4. **Use Examples**: Few-shot improves consistency
   - ✅ Include 2-5 examples of desired input/output

5. **Add Constraints**: Prevent unwanted behavior
   - ✅ "Do not include medication recommendations"
   - ✅ "If uncertain, respond with 'INSUFFICIENT_INFO'"

### Clinical-Specific Tips

1. **Medical Terminology**: LLMs understand clinical terms
   - Use standard abbreviations (STEMI, CHF, COPD)
   - Provide full medication names

2. **Evidence-Based**: Request reasoning
   - ✅ "Cite guidelines (ACC/AHA, GOLD, ADA)"
   - ✅ "Explain clinical reasoning"

3. **Safety**: Add disclaimers
   - ✅ "This is for educational purposes only"
   - ✅ "Final decisions require clinical judgment"

4. **Uncertainty**: Acknowledge limitations
   - ✅ "If diagnosis is unclear, list multiple possibilities"

### Temperature Settings

| Task | Temperature | Why |
|------|-------------|-----|
| Classification | 0.0 | Deterministic, consistent |
| Differential diagnosis | 0.7 | Balanced creativity |
| Patient education | 0.8 | Friendly, varied phrasing |
| Creative brainstorming | 1.0 | Maximum diversity |

---

## 8. Performance Comparison

### Accuracy Benchmarks (Clinical Text Classification)

| Approach | Accuracy | Training Time | Inference Time | Cost per 1K |
|----------|----------|---------------|----------------|-------------|
| TF-IDF + Logistic Regression | 72-78% | <1 min | <1ms | $0 |
| Fine-tuned ClinicalBERT | 88-92% | 15-30 min (GPU) | 50-100ms | $0 (local) |
| LLM Zero-shot (GPT-4 Mini) | 78-85% | No training | 1-2s | $0.30 |
| LLM Few-shot (Claude Haiku) | 85-90% | No training | 1-2s | $0.25 |
| LLM CoT (GPT-4) | 88-92% | No training | 2-4s | $0.60 |

### When to Use Each Approach

**Use LLM Prompting When**:
- ✅ <100 training examples available
- ✅ Rapid prototyping (hours vs. weeks)
- ✅ Task requires reasoning/explanation
- ✅ Multimodal (text + images)
- ✅ Low-volume application (<1K requests/day)

**Use Fine-Tuned Models When**:
- ✅ >1K labeled examples available
- ✅ High-volume production (>10K requests/day)
- ✅ Low latency required (<100ms)
- ✅ Cost-sensitive (millions of requests)
- ✅ On-premise deployment required

**Hybrid Approach** (Best of Both Worlds):
1. Start with LLM prompting (rapid prototyping)
2. Collect user feedback and edge cases
3. Use LLM to generate training data (synthetic labeling)
4. Fine-tune smaller model for production
5. Fall back to LLM for uncertain cases

---

## Key Takeaways

### What We Learned

1. **Zero-Shot Learning**
   - No training examples required
   - 75-85% accuracy on clinical tasks
   - Best for rapid prototyping

2. **Few-Shot Learning**
   - 2-10 examples in prompt
   - +5-15% accuracy over zero-shot
   - Teaches output format

3. **Chain-of-Thought**
   - "Let's think step by step"
   - +10-20% on complex reasoning
   - Transparent, auditable logic

4. **Structured Output**
   - Request JSON/specific format
   - Parse and validate responses
   - Retry on failures

5. **API Options**
   - OpenRouter: Multiple providers, pay-as-you-go
   - HuggingFace: Free tier, medical models
   - Google Vertex: Multimodal, $300 free credits

### Connections to Book Chapters

- **Chapter 8.3**: BERT fine-tuning vs. LLM prompting
- **Chapter 8.5**: RAG enhances LLM with retrieval
- **Chapter 8.7**: Hallucination detection for LLMs
- **Journey 8.8**: Differential diagnosis (will reference Medley at medley.smile.ki.se)

### Clinical Validation Requirements

Before deploying LLM-based systems:

1. **Accuracy** > 85% on held-out test set
2. **Hallucination rate** < 5% (fact-check against knowledge base)
3. **Consistency** > 90% (same input → same output with temp=0)
4. **Clinical expert review** of 100+ outputs
5. **Monitoring**: Track all prompts and responses
6. **Fallback mechanism**: Human review for low-confidence

---

## Exercises

1. **Zero-Shot vs. Few-Shot**: Implement both approaches for medication extraction. Measure accuracy difference. How many examples are needed before diminishing returns?

2. **Chain-of-Thought Engineering**: Design CoT prompts for: (a) Drug-drug interaction checking, (b) Clinical guideline application, (c) Lab value interpretation. Which benefits most from CoT?

3. **Structured Output**: Implement robust JSON parsing with retry logic. Test on 50 LLM responses. What percentage require retries? What are common failure modes?

4. **API Cost Analysis**: Calculate total cost for 10,000 differential diagnoses using: (a) OpenRouter Claude Haiku, (b) OpenRouter GPT-4, (c) HuggingFace BioGPT, (d) Fine-tuned BERT (local). Include training costs.

5. **Prompt Optimization**: Start with basic prompt, then iteratively improve with: (a) Context, (b) Examples, (c) Format specification, (d) Constraints. Measure accuracy at each step.

6. **Multimodal Analysis**: Use Gemini Pro Vision to analyze chest X-rays with clinical context. Compare with radiologist reports. What information does the model miss?

7. **Hallucination Detection**: Generate 100 differential diagnoses. Fact-check against UpToDate or medical textbooks. What's the hallucination rate? Which types of errors occur?

8. **Hybrid System**: Build system that uses LLM for rare cases (<5% of traffic) and fine-tuned BERT for common cases. Measure overall cost and accuracy. What's the optimal threshold?

---

*This notebook is part of "AI in Healthcare" (Volume 1)*  
*Full implementation available in the complete book companion code*